## Introduction

A notebook to analyze the results of exploratory queries ran to understand the data and to determine what KPIs will answer the business problem

## Loading the dataset

In [1]:
from pathlib import Path
import sqlite3

import pandas as pd
import sqlparse

In [2]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [3]:
script_name = '01_initial_checks.sql'
script_path = Path.cwd().parent / "sql" / script_name
script_path

PosixPath('/home/ashish_ubuntu/da_projects/call-center-staffing-analytics/sql/01_initial_checks.sql')

In [4]:
db_name = 'call_center.db'
src_path = Path.cwd().parent / "data/database" / db_name
conn = sqlite3.connect(src_path)


In [5]:
with open(script_path, 'r', encoding='utf-8') as file:
    sql_content = file.read()

queries = sqlparse.split(sql_content)

print(f"Number of queries in script: {len(queries)}")

Number of queries in script: 12


## Reading the output of each query

In [6]:
query = queries[0].strip()
print(query)
result = pd.read_sql_query(query, con=conn)
display(result)

-- DATABASE SCHEMA
-- CREATE TABLE IF NOT EXISTS "call_records" (
-- "call_id" TEXT,
--   "date" DATE,
--   "time" TIME,
--   "agent" TEXT,
--   "department" TEXT,
--   "answered" TEXT,
--   "resolved" TEXT,
--   "speed_of_answer" REAL,
--   "avg_talk_duration" REAL,
--   "satisfaction_rating" REAL
-- );

-- VIEW SCHEMA
-- SELECT
--     *,
--     strftime('%m', date) AS month_num,
--     CASE
--         WHEN strftime('%m', date) = '01' THEN 'January'
--         WHEN strftime('%m', date) = '02' THEN 'February'
--         WHEN strftime('%m', date) = '03' THEN 'March'
--         WHEN strftime('%m', date) = '04' THEN 'April'
--         WHEN strftime('%m', date) = '05' THEN 'May'
--         WHEN strftime('%m', date) = '06' THEN 'June'
--         WHEN strftime('%m', date) = '07' THEN 'July'
--         WHEN strftime('%m', date) = '08' THEN 'August'
--         WHEN strftime('%m', date) = '09' THEN 'September'
--         WHEN strftime('%m', date) = '10' THEN 'October'
--         WHEN strftime('

,call_volume
0,5000


In [7]:
query = queries[1].strip()
print(query)
result = pd.read_sql_query(query, con=conn)
display(result)

-- Call volume by month
SELECT  
    month_num,
    MIN(month_name) AS month_name,
    COUNT(*) AS call_volume
FROM
    vw_call_records_enriched
GROUP BY
    month_num;


,month_num,month_name,call_volume
0,01,January,1772
1,02,February,1616
2,03,March,1612


The variance in total call volume should correlate with the difference in the number of days in each month, however March shows a lower than expected call volume. This warrants an investigation.

In [8]:
query = queries[2].strip()
print(query)
result = pd.read_sql_query(query, con=conn)
display(result)

-- Call Volume by date
SELECT
    date,
    COUNT(*) AS call_volume
FROM
    vw_call_records_enriched
GROUP BY
    date;


,date,call_volume
0,2015-01-01,58
1,2015-01-02,60
2,2015-01-03,58
3,2015-01-04,54
4,2015-01-05,56
5,2015-01-06,58
6,2015-01-07,54
7,2015-01-08,48
8,2015-01-09,68
9,2015-01-10,64


There is a significant drop off in the call volume in the last two days of the dataset, this might have contributed to the lower than expected volume in March.

In [9]:
query = queries[3].strip()
print(query)
result = pd.read_sql_query(query, con=conn)
display(result)

-- Start hour, end hour and total hours for the day
SELECT
    MIN(hour_of_day) AS start_hour,
    MAX(hour_of_day) AS end_hour,
    COUNT(DISTINCT hour_of_day) AS hours_in_a_day
FROM
    vw_call_records_enriched;


,start_hour,end_hour,hours_in_a_day
0,09,18,10


A shift appears to be 9 hours long, however some calls are being handled past 1800. The call volume in 1800 hours seems to be spill over volume rather than a staffed hour

In [10]:
query = queries[4].strip()
print(query)
result = pd.read_sql_query(query, con=conn)
display(result)

-- Call Volume by hour of the day
SELECT
    hour_of_day,
    COUNT(*) AS call_volume
FROM
    vw_call_records_enriched
GROUP BY
    hour_of_day;


,hour_of_day,call_volume
0,09,547
1,10,529
2,11,590
3,12,547
4,13,594
5,14,499
6,15,531
7,16,566
8,17,583
9,18,14


Only 14 calls in 1800 hour opposed to ~500-600 calls on other hours, supports the previous claim of calls handled in 1800 hour being spill over volume

In [11]:
query = queries[5].strip()
print(query)
result = pd.read_sql_query(query, con=conn)
display(result)

-- Call volume by day of week and hour of day
SELECT
    day_of_week_num,
    MIN(day_of_week) AS day_of_week,
    hour_of_day,
    COUNT(*) AS call_volume
FROM
    vw_call_records_enriched
GROUP BY
    day_of_week_num,
    hour_of_day;


,day_of_week_num,day_of_week,hour_of_day,call_volume
0,0,Sunday,09,82
1,0,Sunday,10,66
2,0,Sunday,11,86
3,0,Sunday,12,90
4,0,Sunday,13,100
5,0,Sunday,14,98
6,0,Sunday,15,76
7,0,Sunday,16,66
8,0,Sunday,17,106
9,1,Monday,09,67


No calls handled in 1800 hour on Sunday and only 2-4 calls handled in 1800 hour on other days. Based all of the above observations, I have decided to exclude calls handled 1800 hour for all the hourly analyses

In [12]:
query = queries[6].strip()
print(query)
result = pd.read_sql_query(query, con=conn)
display(result)

-- Call Volume by day of the week
SELECT
    day_of_week_num,
    MIN(day_of_week) AS day_of_week,
    COUNT(*) AS call_volume
FROM
    vw_call_records_enriched
GROUP BY
    day_of_week_num;


,day_of_week_num,day_of_week,call_volume
0,0,Sunday,770
1,1,Monday,675
2,2,Tuesday,679
3,3,Wednesday,712
4,4,Thursday,680
5,5,Friday,768
6,6,Saturday,716


In [13]:
query = queries[7].strip()
print(query)
result = pd.read_sql_query(query, con=conn)
display(result)

-- Average call volume by day of week
WITH daily_calls AS (
    SELECT
        date,
        day_of_week_num,
        MIN(day_of_week) AS day_of_week,
        COUNT(*) AS call_volume
    FROM
        vw_call_records_enriched
    GROUP BY
        date,
        day_of_week_num
)
SELECT
    day_of_week_num,
    MIN(day_of_week) AS day_of_week,
    ROUND(AVG(call_volume), 2) AS avg_call_volume
FROM
    daily_calls
GROUP BY
    day_of_week_num;


,day_of_week_num,day_of_week,avg_call_volume
0,0,Sunday,59.23
1,1,Monday,51.92
2,2,Tuesday,52.23
3,3,Wednesday,59.33
4,4,Thursday,52.31
5,5,Friday,59.08
6,6,Saturday,55.08


Average call volume on each day of week lies within the 50-60 range. The variance is not significant enough to warrant a day of week analysis depth, based on this we can conclude to exclude day of week analysis

In [14]:
query = queries[8].strip()
print(query)
result = pd.read_sql_query(query, con=conn)
display(result)

-- Unanswered call count, answered call count and assigned calls count by agent
SELECT
    agent,
    COUNT(*) FILTER(WHERE answered = 'no') AS unanswered_call_cnt,
    COUNT(*) FILTER(WHERE answered = 'yes') AS answered_call_cnt,
    COUNT(*) AS assigned_call_cnt
FROM
    vw_call_records_enriched
GROUP BY
    agent;


,agent,unanswered_call_cnt,answered_call_cnt,assigned_call_cnt
0,Becky,114,517,631
1,Dan,110,523,633
2,Diane,132,501,633
3,Greg,122,502,624
4,Jim,130,536,666
5,Joe,109,484,593
6,Martha,124,514,638
7,Stewart,105,477,582


Assigned calls range from 582-666. Call assignment seems to be fairly balanced across all agents

In [15]:
query = queries[9].strip()
print(query)
result = pd.read_sql_query(query, con=conn)
display(result)

-- Unanswered call count, answered call count and assigned calls count by department
SELECT
    department,
    COUNT(*) FILTER(WHERE answered = 'no') AS unanswered_call_cnt,
    COUNT(*) FILTER(WHERE answered = 'yes') AS answered_call_cnt,
    COUNT(*) AS assigned_call_cnt
FROM
    vw_call_records_enriched
GROUP BY
    department;


,department,unanswered_call_cnt,answered_call_cnt,assigned_call_cnt
0,Air Conditioner,214,805,1019
1,Fridge,181,795,976
2,Television,175,847,1022
3,Toaster,189,818,1007
4,Washing Machine,187,789,976


With a range of 976-1022, there does not seem to be a skew in the departments the calls are related to

In [16]:
query = queries[10].strip()
print(query)
result = pd.read_sql_query(query, con=conn)
display(result)

-- department coverage by each agent
SELECT
    agent,
    COUNT(DISTINCT department) AS unique_dept_cnt
FROM
    vw_call_records_enriched
GROUP BY
    agent;


,agent,unique_dept_cnt
0,Becky,5
1,Dan,5
2,Diane,5
3,Greg,5
4,Jim,5
5,Joe,5
6,Martha,5
7,Stewart,5


In [17]:
query = queries[11].strip()
print(query)
result = pd.read_sql_query(query, con=conn)
display(result)

-- agent coverage by each department
SELECT
    department,
    COUNT(DISTINCT agent) AS unique_agent_cnt
FROM
    vw_call_records_enriched
GROUP BY
    department;


,department,unique_agent_cnt
0,Air Conditioner,8
1,Fridge,8
2,Television,8
3,Toaster,8
4,Washing Machine,8


All agents are handling calls related to all departments, based on this we can derive KPIs based on Agent and Department comparisons

## Sensitivity Checks

Due to the two outlier dates on March 30 and 31 where the total call volume is lower than the other dates (25-27 against a typical 50-60), a sensitivity check needs to be performed to analyse if there is an impact on the KPIs by removing those dates as outliers.

In [18]:
check_script_name = '02_sensitivity_checks.sql'
check_script_path = Path.cwd().parent / "sql" / check_script_name
check_script_path

PosixPath('/home/ashish_ubuntu/da_projects/call-center-staffing-analytics/sql/02_sensitivity_checks.sql')

In [19]:
with open(check_script_path, 'r', encoding='utf-8') as file:
    check_sql_content = file.read()

check_queries = sqlparse.split(check_sql_content)

print(f"Number of queries in script: {len(check_queries)}")

Number of queries in script: 4


In [20]:
query = check_queries[0].strip()
print(query)
result = pd.read_sql_query(query, con=conn)
display(result)

query = check_queries[1].strip()
print(query)
result = pd.read_sql_query(query, con=conn)
display(result)

-- Purpose: To test the effect of excluding March 30 and March 31 on the final KPI
--          values in order to make a decision on whether to keep the two outliers
--          or exclude them from analysis
-- Source: vw_call_records_enriched (see 00_create_view.sql)
-- Scope: Two KPIs are tested against two sets of data, one for all the dates and
--        another by excluding the two outlier dates of March 30 and 31 where the 
--        volume of calls is lower than the typical daily volume.

-- Unanswered call % by month without excluding the outlier dates
SELECT
    month_num,
    month_name,
    COUNT(*) FILTER(WHERE answered = 'no') AS unanswered_call_cnt,
    COUNT(*) AS total_call_cnt,
    ROUND(
        (CAST(COUNT(*) FILTER(WHERE answered = 'no') AS REAL) / COUNT(*))
        * 100
    , 2) AS unanswered_call_pct
FROM
    vw_call_records_enriched
GROUP BY
    month_num,
    month_name
ORDER BY
    month_num;


,month_num,month_name,unanswered_call_cnt,total_call_cnt,unanswered_call_pct
0,01,January,317,1772,17.89
1,02,February,318,1616,19.68
2,03,March,311,1612,19.29


-- Unanswered call % by month excluding the outlier dates
SELECT
    month_num,
    month_name,
    COUNT(*) FILTER(WHERE answered = 'no') AS unanswered_call_cnt,
    COUNT(*) AS total_call_cnt,
    ROUND(
        (CAST(COUNT(*) FILTER(WHERE answered = 'no') AS REAL) / COUNT(*))
        * 100
    , 2) AS unanswered_call_pct
FROM
    vw_call_records_enriched
WHERE
    date NOT IN ('2015-03-30', '2015-03-31')
GROUP BY
    month_num,
    month_name
ORDER BY
    month_num;


,month_num,month_name,unanswered_call_cnt,total_call_cnt,unanswered_call_pct
0,01,January,317,1772,17.89
1,02,February,318,1616,19.68
2,03,March,295,1560,18.91


Unanswered call % is at 19.29% including the two days, while it's at 18.91% without those dates with a delta of 0.38 percentage points which is about 2% of a 19% rate. From these observations we can conclude the delta is negligible.

In [21]:
query = check_queries[2].strip()
print(query)
result = pd.read_sql_query(query, con=conn)
display(result)

query = check_queries[3].strip()
print(query)
result = pd.read_sql_query(query, con=conn)
display(result)

-- Average speed of answer by month without excluding the outlier dates
SELECT
    month_num,
    month_name,
    COUNT(*) AS answered_call_cnt,
    SUM(speed_of_answer) AS total_speed_of_answer_sec,
    ROUND(AVG(speed_of_answer), 2) AS avg_speed_of_answer_sec
FROM
    vw_call_records_enriched
WHERE 
    answered = 'yes'
GROUP BY
    month_num,
    month_name
ORDER BY
    month_num;


,month_num,month_name,answered_call_cnt,total_speed_of_answer_sec,avg_speed_of_answer_sec
0,01,January,1455,97805.0,67.22
1,02,February,1298,87675.0,67.55
2,03,March,1301,88249.0,67.83


-- Average speed of answer by month excluding the outlier dates
SELECT
    month_num,
    month_name,
    COUNT(*) AS answered_call_cnt,
    SUM(speed_of_answer) AS total_speed_of_answer_sec,
    ROUND(AVG(speed_of_answer), 2) AS avg_speed_of_answer_sec
FROM
    vw_call_records_enriched
WHERE
    date NOT IN ('2015-03-30', '2015-03-31')
    AND
    answered = 'yes'
GROUP BY
    month_num,
    month_name
ORDER BY
    month_num;


,month_num,month_name,answered_call_cnt,total_speed_of_answer_sec,avg_speed_of_answer_sec
0,01,January,1455,97805.0,67.22
1,02,February,1298,87675.0,67.55
2,03,March,1265,85688.0,67.74


Average speed of answer of calls is 67.83 seconds without excluding the outlier days and 67.74 after excluding with a delta of 0.09 second which is about 0.1% of 67s. Due to these observations we can conclude this delta is negligible.

In [22]:
conn.close()

## Conclusions

### Decisions

1. Exclude calls handled in 1800 hour from all hourly KPIs.
2. There is no need to remove the two outlier dates of March 30 and 31 where the total call volume was lower than the average daily volumes as the impact of removing those dates is negligible on the monthly KPIs, the other KPIs where the days are spread across a larger group will have an even smaller impact.
3. Exclude day of week KPIs
4. Agents and departments can be compared directly.

Based on the initial checks of the dataset, I have identified the below set of KPIs as useful to answer the problem statement:

1. unanswered_pct_by_hour
2. unanswered_pct_by_month
3. unanswered_pct_by_agent
4. unanswered_pct_by_department
5. aht_by_agent
6. aht_by_department
7. satisfaction_by_agent
8. satisfaction_by_department
9. speed_of_answer_by_hour
10. speed_of_answer_by_month
11. speed_of_answer_by_agent
12. speed_of_answer_by_department